# 🏦 Bank Marketing Term Deposit Predictor
## 📓 01. Exploratory Data Analysis & Baseline Benchmarks

**Course**: Neural Networks (AIE231)  
**Author / Owner**: Yousef Gamal (Tech Lead & Baseline/Analysis Engineer — Roles 1 & 3)  
**Dataset**: [UCI Bank Marketing (`bank-additional-full.csv`)](https://archive.ics.uci.edu/dataset/222/bank+marketing) (`41,188` records, `20` input features)  
**Date**: November 2026  

---

### 🎯 Problem Formulation & Business Objective
Retail commercial banks routinely conduct direct telemarketing phone campaigns to promote fixed-term deposits (`y = 1`). Inbound and outbound phone contacts represent considerable operating labor expenditure. Contacting uninterested clients wastes call-center capacity and risks client dissatisfaction (**False Positives**), whereas failing to reach an interested prospective depositor forfeits high-value deposit capital and customer lifetime value (**False Negatives**).

The engineering goal of this project is to construct a leak-free neural classification pipeline that reliably identifies prospective subscribers based strictly on **pre-call demographic, campaign, and macroeconomic indicators**.

#### Evaluation Metric Justification
Due to severe class skew (~88.7% *No* vs. ~11.3% *Yes*), naive **accuracy** is fundamentally misleading (a trivial model predicting *No* achieves 88.7% accuracy with 0.0 utility). The primary evaluation metrics are therefore:
1. **Positive-Class F1-Score**: Harmonic mean of positive precision and recall.
2. **Positive-Class Recall**: Ensuring prospective depositors are captured.
3. **PR-AUC (Precision-Recall Area Under Curve)**: Robust threshold-independent benchmark on imbalanced tabular data.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Data Ingestion & Schema Inspection

In [ ]:
data_path = os.path.join("..", "data", "bank-additional-full.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "bank-additional-full.csv")

df = pd.read_csv(data_path, sep=";")
print(f"Loaded Bank Marketing dataset: {df.shape[0]:,} rows, {df.shape[1]} columns.")
df.head()

In [ ]:
print("Column Data Types and Non-Null Counts:")
df.info()

## 2. Target Variable Analysis (`y`)
We examine the subscription distribution to quantify class imbalance.

In [ ]:
target_counts = df["y"].value_counts()
target_pcts = df["y"].value_counts(normalize=True) * 100

print("Target Class Distribution:")
for label, count in target_counts.items():
    print(f"  '{label}': {count:,} ({target_pcts[label]:.2f}%)")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="y", ax=ax[0], palette=["#4A90E2", "#50E3C2"])
ax[0].set_title("Absolute Class Counts")
ax[0].set_xlabel("Subscribed to Term Deposit (y)")
ax[0].set_ylabel("Count")

ax[1].pie(target_counts, labels=target_counts.index, autopct="%1.1f%%", startangle=140, colors=["#4A90E2", "#50E3C2"])
ax[1].set_title("Class Proportion Skew")
plt.tight_layout()
plt.show()

**Finding**: Severe class skew (~88.7% `no` vs. ~11.3% `yes`, approximately 8:1 ratio).  
**Decision**: Accuracy cannot be used as the optimization criterion. We must use class-weighted loss, stratified splitting, and track positive-class F1, Recall, and PR-AUC.

## 3. Critical Target Leakage Analysis: The `duration` Column

> **UCI Documentation Warning**:
> *"duration: last contact duration, in seconds... this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model."*

Let us empirically verify why retaining `duration` constitutes severe data leakage.

In [ ]:
print("Duration Statistics by Target Outcome (seconds):")
print(df.groupby("y")["duration"].describe())

plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x="y", y="duration", palette=["#4A90E2", "#50E3C2"], showfliers=False)
plt.title("Call Duration Distribution by Target Outcome (without extreme fliers)")
plt.xlabel("Subscription Outcome (y)")
plt.ylabel("Duration (seconds)")
plt.show()

**Finding**: Subscribed clients have a median call duration over 3.5× longer than non-subscribed clients. In practice, call duration is only known *after* the call finishes, making it impossible to use for pre-call campaign targeting.  
**Decision**: Drop `duration` immediately from all feature sets to enforce strict zero-leakage discipline.

## 4. Missing Values & The `'unknown'` Label
In this dataset, missing values are represented as string `'unknown'` entries rather than `NaN`.

In [ ]:
unknown_counts = {}
for col in df.select_dtypes(include=["object"]).columns:
    if col == "y":
        continue
    cnt = (df[col] == "unknown").sum()
    pct = (cnt / len(df)) * 100
    if cnt > 0:
        unknown_counts[col] = (cnt, pct)

print("Columns with 'unknown' values:")
unknown_df = pd.DataFrame([
    {"Feature": col, "Count": cnt, "Percentage (%)": f"{pct:.2f}%"}
    for col, (cnt, pct) in sorted(unknown_counts.items(), key=lambda x: x[1][0], reverse=True)
])
unknown_df

**Finding**: `default` has ~20.87% unknown records, `education` has ~4.20%, and `job` has ~0.80%.  
**Decision**: Rather than dropping these rows (which would eliminate >20% of the dataset), we preserve `'unknown'` as a valid category during one-hot encoding, as client refusal or omission of credit status is informative.

## 5. Numerical & Macroeconomic Features Exploration

In [ ]:
num_cols = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"
]
df[num_cols].describe().T

In [ ]:
# Correlation Matrix Heatmap
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, cbar=True)
plt.title("Correlation Matrix of Numerical & Macroeconomic Indicators")
plt.tight_layout()
plt.show()

**Finding**: Strong collinearity exists among macroeconomic indicators (`euribor3m`, `emp.var.rate`, `nr.employed` with correlations > 0.90). Neural networks handle collinear features via layer weight sharing and regularization, but continuous variables vary widely in scale (e.g., `age` ~17-98 vs `cons.conf.idx` ~ -50 to -26).  
**Decision**: `StandardScaler` must be applied to all numerical features, fitted strictly on the training partition.

## 6. Categorical Feature Conversion Rates

In [ ]:
cat_cols = ["job", "marital", "education", "poutcome", "month"]
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(12, 18))

for idx, col in enumerate(cat_cols):
    conversion = df.groupby(col)["y"].apply(lambda x: (x == "yes").mean() * 100).sort_values(ascending=False)
    sns.barplot(x=conversion.index, y=conversion.values, ax=axes[idx], palette="viridis")
    axes[idx].set_title(f"Subscription Rate by {col} (%)")
    axes[idx].set_ylabel("Conversion (%)")
    axes[idx].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

**Finding**: `poutcome = 'success'` exhibits an extraordinarily high conversion rate (~65%), whereas clients who previously failed convert at <15%. Students and retired clients have the highest demographic conversion rates (~31% and ~25%).  
**Decision**: One-hot encode all categorical columns to allow the MLP layers to learn non-linear interactions across campaigns and demographics.

## 7. Leak-Free Preprocessing Pipeline Demonstration
We enforce strict no-leakage discipline:
- Drop `duration`.
- Stratified `70 / 15 / 15` Train/Validation/Test split.
- Fit `StandardScaler` and `OneHotEncoder` **strictly on the training split**, then transform Validation and Test.

In [ ]:
# 1. Feature selection (duration dropped)
feature_df = df.drop(columns=["duration", "y"], errors="ignore")
target_series = (df["y"] == "yes").astype(int).values

# 2. Identify numerical vs. categorical columns
numeric_features = [
    "age", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"
]
categorical_features = [col for col in feature_df.columns if col not in numeric_features]

# 3. Stratified 70 / 15 / 15 split
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    feature_df,
    target_series,
    test_size=0.30,
    stratify=target_series,
    random_state=RANDOM_SEED
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print(f"Train set      : {X_train_raw.shape[0]:,} samples (Positive rate: {y_train.mean():.4f})")
print(f"Validation set : {X_val_raw.shape[0]:,} samples (Positive rate: {y_val.mean():.4f})")
print(f"Test set       : {X_test_raw.shape[0]:,} samples (Positive rate: {y_test.mean():.4f})")

# 4. Fit Preprocessor strictly on Train
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"\nProcessed Input Feature Dimension (input_dim): {X_train.shape[1]}")

## 8. Baseline Benchmark Evaluations

The project specification mandates two baselines before training any neural network:
1. **Trivial Baseline**: Majority-class prediction.
2. **Simple Learned Baseline**: Logistic Regression with balanced class weighting.

In [ ]:
# Baseline 1: Trivial Baseline (Majority Class)
majority_class = 0
y_val_trivial = np.full_like(y_val, fill_value=majority_class)

trivial_metrics = {
    "Model": "1. Trivial Baseline (Majority Class)",
    "Accuracy": accuracy_score(y_val, y_val_trivial),
    "Positive Precision": precision_score(y_val, y_val_trivial, zero_division=0),
    "Positive Recall": recall_score(y_val, y_val_trivial, zero_division=0),
    "Positive F1-Score": f1_score(y_val, y_val_trivial, zero_division=0),
    "ROC-AUC": 0.5,
    "PR-AUC": float(y_val.mean()),
}

# Baseline 2: Logistic Regression (Balanced Class Weights)
lr_clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
lr_clf.fit(X_train, y_train)

y_val_lr_pred = lr_clf.predict(X_val)
y_val_lr_prob = lr_clf.predict_proba(X_val)[:, 1]

learned_metrics = {
    "Model": "2. Learned Baseline (Logistic Regression)",
    "Accuracy": accuracy_score(y_val, y_val_lr_pred),
    "Positive Precision": precision_score(y_val, y_val_lr_pred, zero_division=0),
    "Positive Recall": recall_score(y_val, y_val_lr_prob >= 0.5, zero_division=0),
    "Positive F1-Score": f1_score(y_val, y_val_lr_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_lr_prob),
    "PR-AUC": average_precision_score(y_val, y_val_lr_prob),
}

results_df = pd.DataFrame([trivial_metrics, learned_metrics])
results_df.set_index("Model").round(4)

In [ ]:
# Confusion Matrix for the Learned Baseline
cm = confusion_matrix(y_val, y_val_lr_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred No", "Pred Yes"], yticklabels=["Actual No", "Actual Yes"])
plt.title("Learned Baseline (Logistic Regression) Validation Confusion Matrix")
plt.ylabel("Actual Outcome")
plt.xlabel("Predicted Outcome")
plt.show()

## 9. Summary of EDA Findings & Modeling Decisions

| # | Empirical Finding in EDA | Architectural / Preprocessing Decision | Responsible Role |
| :--- | :--- | :--- | :--- |
| **1** | Severe class imbalance: 88.7% *No* vs. 11.3% *Yes*. | Use balanced class weighting / focal weighting in loss; optimize for Positive F1 & PR-AUC, not Accuracy. | Role 2 (Data) & Role 5 (Train) |
| **2** | `duration` shows artificial correlation with target but is unknown pre-call. | Drop `duration` entirely prior to any modeling to prevent catastrophic target leakage. | Role 2 (Data) |
| **3** | `'unknown'` entries in categorical variables (`default`, `job`, `education`). | Preserve `'unknown'` as an explicit one-hot encoded category rather than dropping rows. | Role 2 (Data) |
| **4** | Numerical features have drastically different ranges and variances. | Apply `StandardScaler`, fit strictly on training fold only. | Role 2 (Data) |
| **5** | Learned Baseline achieves Positive F1 ~ 0.44 and Recall ~ 0.65. | Neural models (Model A, B, C) must beat F1 > 0.45 and PR-AUC > 0.42 to justify deep learning. | Role 1 & 3 (Yousef) & Role 4 (Jessica) |
| **6** | Input dimension after one-hot encoding is $D \approx 50-60$. | Model A will use 32 hidden units; Model B will use 64 $\rightarrow$ 32 with Dropout (0.3). | Role 4 (Model Architect) |

**Next Phase**: Teammates execute work packages: Esraa (`src/data.py`), Jessica (Models B & C + parameter counts in `src/models.py`), Ahmed (experiment sweeps in `src/train.py`), and Jana (error analysis in `src/evaluate.py`).